# Figure 9

With this notebook, we generate Figure 9 in Ronchi et al. (2026). 

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import json

from matplotlib import rc, rcParams

# Set `usetex=False' if you do not have LaTeX installed.
# rc("text", usetex=False)
rc("font", family="serif")
rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

rcParams["mathtext.fontset"] = "stix"
rcParams["font.family"] = "sans-serif"

rcParams["figure.figsize"] = "8.0, 7.0"
rcParams["figure.autolayout"] = True

rcParams["axes.linewidth"] = "1.7"
rcParams["axes.labelpad"] = "15.0"
rcParams["axes.titlepad"] = "15.0"
rcParams["axes.labelsize"] = "30.0"
rcParams["axes.titlesize"] = "30.0"

rcParams["legend.fontsize"] = "30.0"

rcParams["xtick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["xtick.major.pad"] = "10.0"
rcParams["xtick.minor.pad"] = "10.0"
rcParams["xtick.major.size"] = "10.0"
rcParams["xtick.major.width"] = "1.7"
rcParams["xtick.minor.size"] = "5.0"
rcParams["xtick.minor.width"] = "1.7"
rcParams["xtick.labelsize"] = "30"

rcParams["ytick.direction"] = "in"
rcParams["ytick.right"] = True
rcParams["ytick.major.pad"] = "10.0"
rcParams["ytick.minor.pad"] = "10.0"
rcParams["ytick.major.size"] = "10.0"
rcParams["ytick.major.width"] = "1.7"
rcParams["ytick.minor.size"] = "5.0"
rcParams["ytick.minor.width"] = "1.7"
rcParams["ytick.labelsize"] = "30"

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.

    Args:
        stats_path (str): Path to the file where the statistics are saved.

    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []

    with open(stats_path, "r") as json_file:
        data = json.load(json_file)

    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])

    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)

    return mean, std, max_list, min_list

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

training_exp_path = f"{root_path}/tsnpe_experiment_1_maps8_res32_youngxdins"

stats_path = f"{training_exp_path}/data/statistics_train.json"

posterior_samples_path = f"{training_exp_path}/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4"

Load the posterior samples.

In [ ]:
posterior_samples = (
    torch.load(f"{posterior_samples_path}/samples_posterior.pt")
    .detach()
    .cpu()
    .numpy()
)
print(np.shape(posterior_samples))
n_param = np.shape(posterior_samples)[1]

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)
print(np.shape(mean))

In [ ]:
posterior_samples = posterior_samples * std[0:n_param] + mean[0:n_param]

In [ ]:
# Extract 10000 random samples from the posterior distribution.
n_samp = 10000
indices = np.random.choice(
    posterior_samples.shape[0], size=n_samp, replace=False
)
params = posterior_samples[indices]
print(np.shape(params))

Plot the predicted initial magnetic field distribution considering the 1$\sigma$ and 3$\sigma$ uncertainty.

In [ ]:
def pdf_log10_magnetic_field_2normal(
    log10B: np.ndarray,
    mean_1: float,
    sigma_1: float,
    mean_2: float,
    sigma_2: float,
    w: float,
) -> np.ndarray:
    """
    Mixture of two Gaussian distributions for the logarithm log10 of neutron stars' initial magnetic field.
    The means and standard deviations are defined in the configuration file.

    Args:
        log10B (np.ndarray): log10 of the magnetic field strength in [G].

    Returns:
        (np.ndarray): Value of the pdf for each log10B.
    """

    # Define the fractional contribution of the first Gaussian.
    if (w < 0) or (w > 1):
        raise ValueError(
            "The relative weight parameter of the double_log-normal initial magnetic field model "
            "must be in the range 0 and 1."
        )

    pdf_gaussian_1 = (
        1.0
        / (np.sqrt(2 * np.pi) * sigma_1)
        * np.exp(-((log10B - mean_1) ** 2) / (2.0 * sigma_1**2))
    )

    pdf_gaussian_2 = (
        1.0
        / (np.sqrt(2 * np.pi) * sigma_2)
        * np.exp(-((log10B - mean_2) ** 2) / (2.0 * sigma_2**2))
    )

    pdf = w * pdf_gaussian_1 + (1.0 - w) * pdf_gaussian_2

    return pdf

In [ ]:
# Extract the parameters related to the initial magnetic field distribution.
B0_params_list = params[:, 2:7]

In [ ]:
logB0_grid = np.linspace(11, 16, 200)

B0_dist_list = np.zeros((n_samp, len(logB0_grid)))

for i in range(n_samp):
    B0_dist_list[i] = pdf_log10_magnetic_field_2normal(
        logB0_grid,
        B0_params_list[i, 0],
        B0_params_list[i, 1],
        B0_params_list[i, 2],
        B0_params_list[i, 3],
        B0_params_list[i, 4],
    )

In [ ]:
p01 = np.nanpercentile(B0_dist_list, 0.15, axis=0)
p16 = np.nanpercentile(B0_dist_list, 15.85, axis=0)
p50 = np.nanpercentile(B0_dist_list, 50, axis=0)  # median
p84 = np.nanpercentile(B0_dist_list, 84.15, axis=0)
p99 = np.nanpercentile(B0_dist_list, 99.85, axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(logB0_grid, p50, label="Median", linewidth=4, color="tab:purple")

ax.fill_between(
    logB0_grid, p16, p84, alpha=0.4, label=r"$1\sigma$", color="tab:purple"
)
ax.fill_between(
    logB0_grid, p01, p99, alpha=0.2, label=r"$3\sigma$", color="tab:purple"
)

# Adding titles and labels.
plt.xlabel(r"$\log_{10}(B_0 [{\rm Gauss}])$")
plt.ylabel(r"pdf")

# Add grid lines.
plt.grid()

ax.legend(frameon=True, loc=0, prop={"size": 25})

fig.savefig("plots/initialB_youngxdins.pdf", bbox_inches="tight")

# Show the plot.
plt.show()